# Notebook 3: HMM 서사 구조 모델

## 목표
정상 덱의 역할 전이 패턴을 HMM으로 학습.  
추론 시 log-likelihood가 낮은 덱 = 구조적으로 이상한 덱으로 판정.

## 예상 소요 시간
- 학습: 10~20분

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

In [ ]:
!pip install -q hmmlearn
print('설치 완료')

## 1. 시퀀스 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import ast

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_ROLES = 5

seq_df = pd.read_csv(f'{LABELS_DIR}/sequences.csv')
# CSV 저장 시 리스트가 문자열로 저장됨 — 복원
seq_df['sequence'] = seq_df['sequence'].apply(ast.literal_eval)

print(f'시퀀스 수: {len(seq_df)}')
print(f'평균 길이: {seq_df["length"].mean():.1f}장')

# 시퀀스 길이 분포
import matplotlib.pyplot as plt
seq_df['length'].hist(bins=30, figsize=(8, 4))
plt.xlabel('슬라이드 수')
plt.ylabel('덱 수')
plt.title('덱 길이 분포')
plt.show()

## 2. 전이 행렬 시각화 (HMM 학습 전 직관 확인)

In [ ]:
# 역할 전이 빈도 계산
transition_counts = np.zeros((NUM_ROLES, NUM_ROLES), dtype=int)

for seq in seq_df['sequence']:
    for i in range(len(seq) - 1):
        transition_counts[seq[i]][seq[i+1]] += 1

# 정규화
transition_prob = transition_counts / (transition_counts.sum(axis=1, keepdims=True) + 1e-8)

# 히트맵
import seaborn as sns
plt.figure(figsize=(8, 6))
sns.heatmap(
    transition_prob,
    annot=True, fmt='.2f',
    xticklabels=ROLE_NAMES,
    yticklabels=ROLE_NAMES,
    cmap='YlOrRd'
)
plt.xlabel('다음 슬라이드 역할')
plt.ylabel('현재 슬라이드 역할')
plt.title('역할 전이 확률 행렬')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/transition_matrix.png', dpi=100)
plt.show()
print('전이 확률 행렬 저장 완료')

## 3. HMM 학습

Multinomial HMM 사용.  
- 관측값(observation): 슬라이드 역할 (0~4)
- 은닉 상태(hidden state): 덱의 내러티브 단계 (예: 도입부/전개부/결말부)
- 은닉 상태 수는 하이퍼파라미터 — 3~5 탐색

In [ ]:
from hmmlearn import hmm
import pickle
from sklearn.model_selection import train_test_split

# 시퀀스를 hmmlearn 형식으로 변환
# X: concatenated sequences, lengths: each sequence length
def prepare_hmm_data(sequences):
    X = np.concatenate([np.array(s).reshape(-1, 1) for s in sequences])
    lengths = [len(s) for s in sequences]
    return X, lengths


train_seqs, val_seqs = train_test_split(
    seq_df['sequence'].tolist(), test_size=0.15, random_state=42
)

X_train, lengths_train = prepare_hmm_data(train_seqs)
X_val, lengths_val = prepare_hmm_data(val_seqs)

print(f'학습 시퀀스: {len(train_seqs)}개')
print(f'검증 시퀀스: {len(val_seqs)}개')


# 은닉 상태 수 탐색 (3, 4, 5)
best_model = None
best_score = -np.inf
best_n = None
results = []

for n_components in [3, 4, 5]:
    model = hmm.CategoricalHMM(
        n_components=n_components,
        n_iter=100,
        random_state=42,
        verbose=False,
    )
    model.fit(X_train, lengths_train)

    val_score = model.score(X_val, lengths_val) / len(X_val)  # per-step log-likelihood
    results.append({'n_components': n_components, 'val_score': val_score})
    print(f'n_components={n_components} | val log-likelihood/step={val_score:.4f}')

    if val_score > best_score:
        best_score = val_score
        best_model = model
        best_n = n_components

print(f'\n최적 은닉 상태 수: {best_n} (score={best_score:.4f})')

# 최고 모델 저장
with open(f'{MODELS_DIR}/hmm_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('HMM 모델 저장 완료')

## 4. 이상 임계값 결정

정상 덱의 log-likelihood 분포를 보고 이상 판정 임계값을 설정.

In [ ]:
# 전체 덱의 log-likelihood 계산
all_scores = []
for seq in seq_df['sequence'].tolist():
    x = np.array(seq).reshape(-1, 1)
    score = best_model.score(x) / len(seq)  # per-step 정규화
    all_scores.append(score)

scores_np = np.array(all_scores)

# 분포 시각화
plt.figure(figsize=(10, 4))
plt.hist(scores_np, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
plt.xlabel('Log-Likelihood per Step')
plt.ylabel('덱 수')
plt.title('정상 덱의 Log-Likelihood 분포')

# 하위 5%를 이상 임계값으로 설정
threshold_5pct = np.percentile(scores_np, 5)
threshold_10pct = np.percentile(scores_np, 10)
plt.axvline(threshold_5pct, color='red', linestyle='--', label=f'5th percentile: {threshold_5pct:.3f}')
plt.axvline(threshold_10pct, color='orange', linestyle='--', label=f'10th percentile: {threshold_10pct:.3f}')
plt.legend()
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_score_distribution.png', dpi=100)
plt.show()

print(f'5th percentile threshold: {threshold_5pct:.4f}')
print(f'10th percentile threshold: {threshold_10pct:.4f}')

# 임계값 저장
import json
thresholds = {
    'threshold_5pct': float(threshold_5pct),
    'threshold_10pct': float(threshold_10pct),
    'mean': float(scores_np.mean()),
    'std': float(scores_np.std()),
}
with open(f'{MODELS_DIR}/hmm_thresholds.json', 'w') as f:
    json.dump(thresholds, f, indent=2)
print('임계값 저장 완료')

## 5. HMM 해석 — 학습된 전이 행렬 확인

In [ ]:
# 학습된 방출 확률 (emission probability)
# 각 은닉 상태가 어떤 역할에 주로 대응되는지 확인
emission_df = pd.DataFrame(
    best_model.emissionprob_,
    columns=ROLE_NAMES,
    index=[f'은닉상태 {i}' for i in range(best_n)]
)
print('방출 확률 (은닉 상태 → 관측 역할):')
print(emission_df.round(3))

# 은닉 상태 해석 (가장 높은 방출 역할로 이름 붙이기)
state_names = []
for i in range(best_n):
    dominant_role = ROLE_NAMES[emission_df.iloc[i].argmax()]
    state_names.append(f'State{i}({dominant_role})')

# 전이 확률 히트맵
plt.figure(figsize=(6, 5))
sns.heatmap(
    best_model.transmat_,
    annot=True, fmt='.2f',
    xticklabels=state_names,
    yticklabels=state_names,
    cmap='Blues'
)
plt.title('HMM 은닉 상태 전이 확률')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/hmm_transition.png', dpi=100)
plt.show()

In [ ]:
print('=== Notebook 3 완료 ===')
print(f'HMM 모델: {MODELS_DIR}/hmm_model.pkl')
print(f'이상 임계값: {MODELS_DIR}/hmm_thresholds.json')
print(f'최적 은닉 상태 수: {best_n}')
print(f'Val log-likelihood/step: {best_score:.4f}')
print('\nNotebook 4 (최종 평가)로 이동하세요.')